# ErrorGnoMark Randomized Benchmarking (RB) Tutorial
**Comprehensive Guide to Modular RB Architecture (v5.4)**

This tutorial demonstrates the full capabilities of the RB module in ErrorGnoMark. We adopt a **Top-Down approach**:
1.  **Quick Start**: Running Standard RB (1Q & 2Q) immediately using the Functional API.
2.  **Advanced**: Running Interleaved RB (IRB) for gate calibration.
3.  **Under the Hood**: Understanding circuit decomposition, topology mapping, and data protocols.
4.  **Automation**: Using the high-level Class API for one-line experiment execution.

---
### 🛠️ Setup & Initialization

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Any

# --- Framework Imports ---
from egm.execution.executor import Executor
from egm.foundation.backends.dummy_backend import DummyBackend
from egm.foundation.circuits.circuit import QuantumCircuit, Gate

# --- RB Module Imports (Functional & Class API) ---
from egm.experiments.physical.benchmarking.rb import (
    # Single Circuit Generators (for visualization)
    generate_single_standard_rb_circuit,
    generate_single_interleaved_rb_circuit,
    # Batch Generators (for execution)
    generate_standard_rb_circuits,
    generate_respectively_standard_rb_circuits,
    generate_simultaneously_standard_rb_circuits,
    generate_interleaved_rb_circuits,
    generate_respectively_interleaved_rb_circuits,
    generate_simultaneously_interleaved_rb_circuits,
    # Class API
    StandardRBExperiment,
    InterleavedRBExperiment,
)

# --- Analysis Imports ---
from egm.analysis.rb import (
    stitch_rb_results,
    analyze_rb_standard,
    analyze_rb_simultaneous,
    calculate_epg,
)

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

backend = DummyBackend(clifford_fidelity=0.995, spam_error_rate=0.01)
engine = Executor(backend=backend)

SEED = 2025
SHOTS = 1024
DEPTHS = [2, 4, 8, 16, 32]
CIRCUITS_PER_DEPTH = 3  # kept low for demo speed

print("Environment configured. Executor ready.")

# 1. Quick Start: Standard RB (Functional API)

In this section, we manually generate, execute, and analyze Standard RB experiments.
We will cover **Single Qubit (1Q)** and **Two Qubit (2Q)** scenarios, demonstrating both **Respective** (Isolated) and **Simultaneous** (Parallel) execution modes.

## 1.1 Single Qubit (1Q) Standard RB
First, let's look at what a single RB circuit looks like.

In [ ]:
# --- Level 2 Demo: Visualize a Single 1Q Circuit ---
print("--- [Visual Demo] Single 1Q Standard RB Circuit ---")
demo_1q_circuit = generate_single_standard_rb_circuit(
    qubits=[0],
    depth=4,
    seed=SEED,
    native_gates=['rx', 'rz', 'cz'] # Force physical decomposition
)
demo_1q_circuit.draw(style="text")

In [ ]:
# --- Respective Execution: Run RB on Qubit 0 ---
print("\n--- [Execution] 1Q Respective RB (Qubit 0) ---")

# 1. Generate Batch
circuits_1q = generate_standard_rb_circuits(
    qubits=[0],
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    native_gates=['rx', 'rz', 'cz']
)

# 2. Execute
# Returns List[Tuple(ideal_probs, noisy_counts)]
raw_results_1q = engine.execute_with_ideal(circuits_1q, shots=SHOTS)

# 3. Stitch Data (Combine Counts + Metadata)
stitched_1q = stitch_rb_results(circuits_1q, raw_results_1q)

# 4. Analyze
result_1q = analyze_rb_standard(stitched_1q)

if result_1q.success:
    p_val = next(p.value for p in result_1q.fit.params if p.name == 'p')
    print(f"✅ Fit Success! Decay (p): {p_val:.5f} | EPC: {result_1q.notes}")
else:
    print(f"❌ Fit Failed: {result_1q.error_message}")

In [ ]:
# --- Simultaneous Execution: Run RB on Qubit 0 and Qubit 1 in Parallel ---
print("\n--- [Execution] 1Q Simultaneous RB (Q0 || Q1) ---")

groups_1q = [[0], [1]]

# 1. Generate Simultaneous Batch
# Note: Returns Dict[Depth, List[Circuits]]
simul_map_1q = generate_simultaneously_standard_rb_circuits(
    qubit_groups=groups_1q,
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    native_gates=['rx', 'rz', 'cz']
)

# Flatten for execution
all_simul_1q = [c for d in DEPTHS for c in simul_map_1q[d]]

# 2. Execute
raw_res_simul_1q = engine.execute_with_ideal(all_simul_1q, shots=SHOTS)

# 3. Stitch
stitched_simul_1q = stitch_rb_results(all_simul_1q, raw_res_simul_1q)

# 4. Analyze (Auto-infers groups)
results_dict_1q = analyze_rb_simultaneous(stitched_simul_1q)

for g, res in results_dict_1q.items():
    p_val = next(p.value for p in res.fit.params if p.name == 'p')
    print(f"Group {g}: p={p_val:.5f} (EPC={res.notes})")

## 1.2 Two Qubit (2Q) Standard RB
Now let's scale up to 2-qubit gates (Clifford group size increases significantly).

In [ ]:
# --- Level 2 Demo: Visualize a Single 2Q Circuit ---
print("--- [Visual Demo] Single 2Q Standard RB Circuit ---")
demo_2q_circuit = generate_single_standard_rb_circuit(
    qubits=[0, 1],
    depth=2, # Keep depth low for readability
    seed=SEED,
    native_gates=['rx', 'rz', 'cz']
)
# Notice the decomposition into native CZ/RX/RZ gates
demo_2q_circuit.draw(style="text")

In [ ]:
# --- Respective Execution: Run RB on Pair [0, 1] ---
print("\n--- [Execution] 2Q Respective RB ([0, 1]) ---")
circuits_2q = generate_standard_rb_circuits(
    qubits=[0, 1],
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED,
    native_gates=['rx', 'rz', 'cz']
)
raw_res_2q = engine.execute_with_ideal(circuits_2q, shots=SHOTS)
fit_2q = analyze_rb_standard(stitch_rb_results(circuits_2q, raw_res_2q))
print(f"2Q Respective Fit: Success={fit_2q.success}, EPC={fit_2q.notes}")


# --- Simultaneous Execution: Run [0, 1] and [2, 3] in Parallel ---
print("\n--- [Execution] 2Q Simultaneous RB ([0, 1] || [2, 3]) ---")
groups_2q = [[0, 1], [2, 3]]

simul_map_2q = generate_simultaneously_standard_rb_circuits(
    qubit_groups=groups_2q,
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED + 1,
    native_gates=['rx', 'rz', 'cz']
)
all_simul_2q = [c for d in DEPTHS for c in simul_map_2q[d]]

raw_res_simul_2q = engine.execute_with_ideal(all_simul_2q, shots=SHOTS)
results_dict_2q = analyze_rb_simultaneous(stitch_rb_results(all_simul_2q, raw_res_simul_2q))

for g, res in results_dict_2q.items():
    print(f"Group {g}: Success={res.success}, EPC={res.notes}")

# 2. Advanced Application: Interleaved RB (Functional API)

Interleaved RB (IRB) is used to estimate the specific error of a target gate (Error Per Gate - EPG).
We interleave the target gate between every Clifford gate.

## 2.1 1Q Interleaved RB (Target: X Gate)

In [ ]:
# Define Target Gate
target_gate_1q = Gate("x", (0,))

# --- Level 2 Demo: Visualize Interleaved Circuit ---
print("--- [Visual Demo] Single 1Q Interleaved Circuit (Target: X) ---")
demo_irb_1q = generate_single_interleaved_rb_circuit(
    qubits=[0],
    depth=3,
    interleaved_gate=target_gate_1q,
    seed=SEED,
    native_gates=['rx', 'rz', 'cz']
)
# You should see the target gate (decomposed) appearing frequently
demo_irb_1q.draw(style="text")

In [ ]:
# --- Respective IRB ---
print("\n--- [Execution] 1Q Respective IRB (Target: X) ---")
# 1. Generate Reference & Interleaved Batches
c_ref = generate_standard_rb_circuits(
    qubits=[0], depths=DEPTHS, circuits_per_depth=CIRCUITS_PER_DEPTH, seed=SEED
)
c_int = generate_interleaved_rb_circuits(
    qubits=[0], depths=DEPTHS, circuits_per_depth=CIRCUITS_PER_DEPTH,
    interleaved_gate=target_gate_1q, seed=SEED
)

# 2. Execute & Stitch
stitched_ref = stitch_rb_results(c_ref, engine.execute_with_ideal(c_ref, shots=SHOTS))
stitched_int = stitch_rb_results(c_int, engine.execute_with_ideal(c_int, shots=SHOTS))

# 3. Analyze
fit_ref = analyze_rb_standard(stitched_ref)
fit_int = analyze_rb_standard(stitched_int)

# 4. Calculate EPG
if fit_ref.success and fit_int.success:
    p_ref = next(p.value for p in fit_ref.fit.params if p.name == 'p')
    p_int = next(p.value for p in fit_int.fit.params if p.name == 'p')
    epg = calculate_epg(p_ref, p_int, num_qubits=1)
    print(f"Calculated EPG for X gate: {epg:.5e}")


# --- Simultaneous IRB (Stress Test: X on Q0 || X on Q1) ---
print("\n--- [Execution] 1Q Simultaneous IRB ---")
# Using the SAME template gate 'X on Q0'. The function automatically remaps it to Q1 for the second group.
simul_map_int_1q = generate_simultaneously_interleaved_rb_circuits(
    qubit_groups=[[0], [1]],
    depths=DEPTHS, circuits_per_depth=CIRCUITS_PER_DEPTH,
    interleaved_gate=target_gate_1q, # Defined on (0,), will be mapped to (1,)
    seed=SEED+5
)
# Execution skipped for brevity, but follows same pattern as Standard Simultaneous RB
print("Simultaneous IRB circuits generated successfully.")

## 2.2 2Q Interleaved RB (Target: CZ Gate)
Calibrating two-qubit gates is critical. Here we test a CZ gate.

In [ ]:
# Define Target Gate
target_gate_2q = Gate("cz", (0, 1))

# --- Level 2 Demo: Visualize 2Q Interleaved Circuit ---
print("--- [Visual Demo] Single 2Q Interleaved Circuit (Target: CZ) ---")
demo_irb_2q = generate_single_interleaved_rb_circuit(
    qubits=[0, 1],
    depth=2,
    interleaved_gate=target_gate_2q,
    seed=SEED,
    # No decomposition here to see the abstract 'CZ' clearly in the printout
    native_gates=None
)
demo_irb_2q.draw(style="text")

In [ ]:
print("\n--- [Execution] 2Q Respective IRB (Target: CZ) ---")

# Generate
c_ref_2q = generate_standard_rb_circuits([0, 1], DEPTHS, CIRCUITS_PER_DEPTH, seed=SEED)
c_int_2q = generate_interleaved_rb_circuits([0, 1], DEPTHS, CIRCUITS_PER_DEPTH, interleaved_gate=target_gate_2q, seed=SEED)

# Execute
res_ref = engine.execute_with_ideal(c_ref_2q, shots=SHOTS)
res_int = engine.execute_with_ideal(c_int_2q, shots=SHOTS)

# Analyze
fit_ref_2q = analyze_rb_standard(stitch_rb_results(c_ref_2q, res_ref))
fit_int_2q = analyze_rb_standard(stitch_rb_results(c_int_2q, res_int))

# EPG
if fit_ref_2q.success and fit_int_2q.success:
    p_ref = next(p.value for p in fit_ref_2q.fit.params if p.name == 'p')
    p_int = next(p.value for p in fit_int_2q.fit.params if p.name == 'p')
    epg = calculate_epg(p_ref, p_int, num_qubits=2)
    print(f"Calculated EPG for CZ gate: {epg:.5e}")

# 3. Under the Hood: Mechanics & Protocols

Now that we have run experiments, let's explore the underlying mechanisms that make this possible: **Decomposition**, **Topology Mapping**, and **Data Stitching**.

In [ ]:
from egm.foundation.circuits.circuit import QuantumCircuit, Gate

print("--- 3.1 QuantumCircuit.decompose ---")
# 1. Create a logical circuit with abstract Clifford gates
logical = QuantumCircuit(qubits=[0])
logical.add_gates([Gate("h", (0,)), Gate("y", (0,))])
print("Logical Circuit:")
logical.draw()

# 2. Decompose into specific basis
physical = logical.decompose(basis_gates=['rx', 'rz'])
print("\nPhysical Circuit (Basis: rx, rz):")
physical.draw()

In [ ]:
print("\n--- 3.3 Data Protocol: stitch_rb_results ---")
# Demonstrating the format expected by the Analyzer
# Input: List[Circuit] + List[Tuple(Ideal, Noisy)]

fake_circuits = [QuantumCircuit(qubits=[0])]
fake_circuits[0].metadata = {"depth": 10} # Metadata is CRUCIAL

# Simulate Engine Output: (Ideal_Dict, Noisy_Dict)
fake_execution = [
    (None, {"0": 90, "1": 10})
]

stitched = stitch_rb_results(fake_circuits, fake_execution)
print("Stitched Data Output:")
print(stitched)
# This list of dicts is what analyze_rb_standard receives.

# 4. Final Solution: Class-Based Automation

The `Experiment` classes wrap all previous steps (Generation -> Execution -> Stitching -> Analysis) into a single object managed lifecycle.

In [ ]:
print("--- 4.1 StandardRBExperiment Class ---")

exp = StandardRBExperiment(
    qubits=[0],
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED + 999,
    native_gates=['rx', 'rz', 'cz'],
)

result = exp.run(engine, shots=SHOTS, plot=False)

print(f"Class Result Success: {result.success}")
print(f"Result Notes: {result.notes}")

In [ ]:
print("\n--- 4.2 InterleavedRBExperiment Class ---")

# 1. Initialize with target gate
irb_exp = InterleavedRBExperiment(
    interleaved_gate=Gate("x", (0,)),
    qubits=[0],
    depths=DEPTHS,
    circuits_per_depth=CIRCUITS_PER_DEPTH,
    seed=SEED + 888
)

# 2. Run Pipeline
irb_result = irb_exp.run(engine, shots=SHOTS, plot=False)

print(f"IRB Class Success: {irb_result['success']}")
if irb_result['success']:
    print(f"EPG: {irb_result['epg']:.5e}")

## Summary
We have successfully demonstrated:
1.  **Level 2 Generators**: Creating individual logical/physical circuits.
2.  **Functional API**: Manually composing batches for Respective and Simultaneous experiments.
3.  **Core Mechanics**: How decomposition and data stitching connect the layers.
4.  **Class API**: The simplified interface for end-users.